# Analysis of the Gun Violence Dataset

# Part 1. &ndash; Problem Understanding & Data Understanding

Authors: Kacper Neumann, Stanisław Apanasiewicz, Jan Matusiak

----

## 1. The Dataset

### 1.1 General Information

The [Gun Violence Data](https://www.kaggle.com/datasets/jameslko/gun-violence-data) dataset contains detailed information about gun violence incidents in the United States, available in CSV format. It is based on records collected by the [Gun Violence Archive (GVA)](https://www.gunviolencearchive.org/), an independent organization created to provide a comprehensive database incidents in the U.S.

The CSV file contains data for all recorded gun violence incidents in the U.S. between January 2013 and March 2018.

In [117]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download('jameslko/gun-violence-data')
df = pd.read_csv(os.path.join(path, 'gun-violence-data_01-2013_03-2018.csv'))

In [118]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 239677 entries, 0 to 239676
Data columns (total 29 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   incident_id                  239677 non-null  int64  
 1   date                         239677 non-null  str    
 2   state                        239677 non-null  str    
 3   city_or_county               239677 non-null  str    
 4   address                      223180 non-null  str    
 5   n_killed                     239677 non-null  int64  
 6   n_injured                    239677 non-null  int64  
 7   incident_url                 239677 non-null  str    
 8   source_url                   239209 non-null  str    
 9   incident_url_fields_missing  239677 non-null  bool   
 10  congressional_district       227733 non-null  float64
 11  gun_stolen                   140179 non-null  str    
 12  gun_type                     140226 non-null  str    
 13  incident_c

In [119]:
df['incident_id'].nunique()

239677

There are 239677 samples described with 29 attributes each. There are no duplicate rows.

In [ ]:
missing_stats = pd.DataFrame({
    'column': df.columns,
    'n_missing': [df[col].isna().sum() for col in df.columns],
})
missing_stats['%_missing'] = (missing_stats['n_missing'] / df.shape[0]).apply(lambda x: f'{x:.0%}')
missing_stats

,column,n_missing,%_missing
0,incident_id,0,0%
1,date,0,0%
2,state,0,0%
3,city_or_county,0,0%
4,address,16497,7%
5,n_killed,0,0%
6,n_injured,0,0%
7,incident_url,0,0%
8,source_url,468,0%
9,incident_url_fields_missing,0,0%


### 1.2 Attributes

| Attribute | Type | Meaning | Unit / Format | Remarks |
|---|---|---|---|---|
| `incident_id` | Nominal | Unique identifier assigned to the incident | Positive integer ID |  |
| `date` | Ordinal | Date of the incident | Date in `YYYY-MM-DD` format |  |
| `state` | Nominal | U.S. state where the incident occurred | String |  |
| `city_or_county` | Nominal | City or county of the crime | String |  |
| `address` | Nominal | Address of the location | String | `NaN` for ~7% of rows |
| `n_killed` | Numerical | Number of people killed | Non-negative integer count |  |
| `n_injured` | Numerical | Number of people injured | Non-negative integer count |  |
| `incident_url` | Nominal | URL to the Gun Violence Archive incident page | URL string |  |
| `source_url` | Nominal | URL to the reporting source | URL string | `NaN` for <1% of rows |
| `incident_url_fields_missing` | Nominal | Flag indicating whether `incident_url` is missing | Boolean | Insignificant since `incident_url` is present in all rows, can be ignored |
| `congressional_district` | Nominal | ID of the congressional district connected to the location | Non-negative integer ID | `NaN` for ~5% of rows |
| `gun_stolen` | Nominal | Status of the guns used (e.g. `Stolen`) | Multi-valued string encoded as `index::status` pairs delimited by `\|\|` | `NaN` for ~42% of rows, missing mostly in the same rows as `gun_type`, requires preprocessing |
| `gun_type` | Nominal | Types of the guns used (e.g. `Handgun`) | Multi-valued string encoded as `index::type` pairs delimited by `\|\|` | `NaN` for ~41% of rows, missing in the same rows as `gun_stolen` and `n_guns_involved`, may be a specific firearm model or a more general category, requires preprocessing |
| `incident_characteristics` | Nominal | Details of the incident (e.g. victims, guns) | Multi-valued string delimited by `\|\|` | `NaN` in <1% of rows, some categories repeat across multiple thousands of rows while others are unique, requires preprocessing |
| `latitude` | Numerical | Geographic latitude | Degrees with 4-decimal points precision (float) | `NaN` for ~3% of rows, missing in the same rows as `longitude` |
| `location_description` | Nominal | Location details (e.g. name of the grocery store involved) | String | `NaN` for ~82% of rows, some categories repeat while others are unique, likely insignificant for further research |
| `longitude` | Numerical | Geographic longitude | Degrees with 4-decimal points precision (float) | `NaN` for ~3% of rows, missing in the same rows as `latitude` |
| `n_guns_involved` | Numerical | Number of guns involved | Positive integer count | `NaN` for ~41% of rows, missing in the same rows as `gun_type` |
| `notes` | Nominal | Additional information about the crime, may provide similar information to `incident_characteristics` | String | `NaN` for ~34% of rows, some values have similar structure but are mostly unique, may be difficult to preprocess |
| `participant_age` | Numerical | Age of the participants at the time of the incident | Multi-valued string encoded as `index::age` pairs delimited by `\|\|` | `NaN` for ~39% of rows, requires preprocessing |
| `participant_age_group` | Ordinal | Discretized `participant_age` (e.g. `Child 0-11`) | Multi-valued string encoded as `index::group` pairs delimited by `\|\|` | `NaN` for ~18% of rows, missing mostly in the same rows as `participant_age`, requires preprocessing |
| `participant_gender` | Nominal | Gender of the participants (`Male` / `Female`) | Multi-valued string encoded as `index::gender` pairs delimited by `\|\|` | `NaN` for ~15% of rows, requires preprocessing |
| `participant_name` | Nominal | Full name of the participants | Multi-valued string encoded as `index::name` pairs delimited by `\|\|` | `NaN` for ~51% of rows, requires preprocessing |
| `participant_relationship` | Nominal | Relationship between the participants (e.g. `Co-worker`) | Multi-valued string encoded as `index::relationship` pairs delimited by `\|\|` | `NaN` for ~93% of rows, likely insignificant for further reaserch |
| `participant_status` | Nominal | Status of the participants (e.g. `Killed`) | Multi-valued string encoded as `index::status` pairs delimited by `\|\|` | `NaN` for ~12% of rows, requires preprocessing |
| `participant_type` | Nominal | Type of the participants (e.g. `Victim`) | Multi-valued string encoded as `index::type` pairs delimited by `\|\|` | `NaN` for ~10% of rows, requires preprocessing |
| `sources` | Nominal | Similar to `source_url`, may include multiple sources | Multi-valued string of URLs delimited by `\|\|` | `NaN` for <1% of rows, requires preprocessing |
| `state_house_district` | Nominal | Voting house district | Positive integer ID | `NaN` for ~16% of rows, missing mostly in the same rows as `state_senate_district` |
| `state_senate_district` | Nominal | Voting senate district | Positive integer ID | `NaN` for ~13% of the datase, missing mostly in the same rows as `state_house_district` |

## 2. Data Mining Goals & Success Criteria

TODO